# Fairness extension on Lee 2019 — within-cohort heterogeneity + optional demographic stratification

Reads the compact Lee 2019 cache and runs:

  1. Per-subject A1 attack accuracy across all 54 subjects for FBCSP, Riemann, and EEGNet victims (within-session train/test split).
  2. Heterogeneity statistics: mean, std, IQR, decile gap.
  3. Demographic stratification (sex / age / handedness) IF `data/external/lee2019_demographics.tsv` is populated. Lee 2019 publishes only cohort aggregates, so in the default state this branch reports "demographic stratification skipped" cleanly.

Prerequisite: `A3_lee2019_download.ipynb` must have completed.

**Runtime → L4 GPU.** Expected wall: ~30-45 min.

**Don't `Save a copy in GitHub` from Colab.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
import os
os.environ['BCI_LEE2019_CACHE'] = '/content/drive/MyDrive/bci_cache'
import glob, sys
WIN = '/content/drive/MyDrive/bci_cache/lee2019_mi/windowed'
files = sorted(glob.glob(os.path.join(WIN, '*.npz')))
print(f'Cached: {len(files)}/108')
if len(files) < 108:
    sys.exit('!!! cache incomplete; run A3_lee2019_download.ipynb first.')

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available())

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich scipy

In [ ]:
!BCI_LEE2019_CACHE=/content/drive/MyDrive/bci_cache PYTHONUNBUFFERED=1 \
  python -u -m experiments.32_fairness_lee2019 --all

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
RESULTS = 'results/32_fairness_lee2019.json'
TAG = 'sf_lee2019'
try:
    git_sha = subprocess.check_output(['git','rev-parse','HEAD']).decode().strip()
except Exception: git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': 'experiments.32_fairness_lee2019',
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS]}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json','w') as f: json.dump(meta, f, indent=2)
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing.')
print('--- BEGIN 32_fairness_lee2019.json ---')
print(open(RESULTS).read())
print('--- END 32_fairness_lee2019.json ---')
print()
print('--- BEGIN run meta ---')
print(json.dumps(meta, indent=2))
print('--- END run meta ---')